In [21]:
#1 — Imports
from pathlib import Path
from datetime import UTC, datetime
import hashlib
import pickle
import re

from langchain_core.documents import Document

In [22]:
#2 — Project Paths
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"

ARTIFACTS = PROJECT_ROOT / "artifacts"

ARTIFACTS.mkdir(exist_ok=True)

In [23]:
#3 – Helper Functions
def sha256(text: str) -> str:
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

In [24]:
#4 – Parse Front Matter
def parse_frontmatter(text: str):
    """
    Extract YAML frontmatter if present.
    """

    if not text.startswith("---"):
        return {}, text

    parts = text.split("---", 2)

    if len(parts) < 3:
        return {}, text

    yaml_block = parts[1]
    body = parts[2]

    metadata = {}

    for line in yaml_block.splitlines():

        if ":" in line:

            key, value = line.split(":", 1)

            metadata[key.strip()] = value.strip()

    return metadata, body.strip()    

In [25]:
#5 – Markdown Cleaning
def clean_markdown(text: str) -> str:
    """
    Clean markdown while preserving semantic content.
    """

    if not text:
        return ""

    # Remove fenced code blocks
    text = re.sub(
        r"```.*?```",
        " ",
        text,
        flags=re.DOTALL
    )

    # Inline code
    text = re.sub(
        r"`([^`]*)`",
        r"\1",
        text
    )

    # Images
    text = re.sub(
        r"!\[.*?\]\(.*?\)",
        " ",
        text
    )

    # Links
    text = re.sub(
        r"\[(.*?)\]\((.*?)\)",
        r"\1",
        text
    )

    # Markdown headings
    text = re.sub(
        r"^#+\s*",
        "",
        text,
        flags=re.MULTILINE
    )

    # Blockquotes
    text = re.sub(
        r"^>\s*",
        "",
        text,
        flags=re.MULTILINE
    )

    # Horizontal rules
    text = re.sub(
        r"^-{3,}$",
        "",
        text,
        flags=re.MULTILINE
    )

    # Bullet lists
    text = re.sub(
        r"^\s*[-*+]\s+",
        "",
        text,
        flags=re.MULTILINE
    )

    # Numbered lists
    text = re.sub(
        r"^\s*\d+\.\s+",
        "",
        text,
        flags=re.MULTILINE
    )

    text = re.sub(r"\n+", "\n", text)

    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [26]:
#6 – Build Documents
documents: list[Document] = []

for path in sorted(DATA_DIR.rglob("*.md")):
    
    raw_text = path.read_text(encoding="utf-8", errors="replace")
    frontmatter, body = parse_frontmatter(raw_text)
    cleaned_text = clean_markdown(body)
    relative = path.relative_to(DATA_DIR)
    parts = relative.parts
    company = parts[0].lower()
    product_area = (parts[1].replace("-", "_") if len(parts) > 1 else "general_support")

    metadata = {

        "document_id": sha256(str(relative)),
        "document_hash": sha256(cleaned_text),
        "company": company,
        "product_area": product_area,
        "title": frontmatter.get("title",path.stem),
        "url": (frontmatter.get("source_url") or frontmatter.get("final_url") or ""),
        "file_name": path.name,
        "source_path": str(relative),
        "extension": path.suffix,
        "source_type": "markdown",
        "schema_version": "1.0",
        "indexed_at": datetime.now(UTC).isoformat(),
        "last_modified": datetime.fromtimestamp(path.stat().st_mtime,tz=UTC).isoformat()
        
    }
    documents.append(Document(page_content=cleaned_text, metadata=metadata)
    )

In [27]:
#7 – Verify
print(f"Documents Loaded : {len(documents)}")

Documents Loaded : 773


In [28]:
#8 – Inspect Document
sample = documents[0]
print(sample)

page_content='How do I learn more about Amazon and Anthropic’s strategic collaboration?
_Last updated: 2026-03-16T20:39:36Z_
Learn more by viewing this press release.
Related Articles
How do I get access to Claude in Amazon Bedrock?
How can I learn more about Claude API pricing?
How do I pay for my Claude API usage?
How can I access the personal information that Anthropic has on my account?
Where can I learn more about Anthropic's Privacy practices?' metadata={'document_id': 'bac5cafbe3fad78362bf9f7102a348a004454d67a65cec3cf592deed2ad6d9fb', 'document_hash': 'f95538b822db9ca84075a7b73f7b4ed08c2e4e087ec3c972c08f9e6565a85924', 'company': 'claude', 'product_area': 'amazon_bedrock', 'title': '"How do I learn more about Amazon and Anthropic’s strategic collaboration?"', 'url': '"https://support.claude.com/en/articles/10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration"', 'file_name': '10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collabora

In [29]:
#9 – Inspect Metadata
sample.metadata

{'document_id': 'bac5cafbe3fad78362bf9f7102a348a004454d67a65cec3cf592deed2ad6d9fb',
 'document_hash': 'f95538b822db9ca84075a7b73f7b4ed08c2e4e087ec3c972c08f9e6565a85924',
 'company': 'claude',
 'product_area': 'amazon_bedrock',
 'title': '"How do I learn more about Amazon and Anthropic’s strategic collaboration?"',
 'url': '"https://support.claude.com/en/articles/10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration"',
 'file_name': '10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration.md',
 'source_path': 'claude\\amazon-bedrock\\10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration.md',
 'extension': '.md',
 'source_type': 'markdown',
 'schema_version': '1.0',
 'indexed_at': '2026-07-07T20:51:17.786235+00:00',
 'last_modified': '2026-07-02T21:53:33.772476+00:00'}

In [30]:
#10 – Document Statistics
from collections import Counter

company_counter = Counter(
    doc.metadata["company"]
    for doc in documents
)

company_counter

Counter({'hackerrank': 438, 'claude': 321, 'visa': 14})

In [31]:
#11 – Product Area Statistics
product_counter = Counter(
    doc.metadata["product_area"]
    for doc in documents
)

product_counter

Counter({'integrations': 94,
         'screen': 88,
         'claude': 83,
         'library': 51,
         'settings': 51,
         'team_and_enterprise_plans': 48,
         'hackerrank_community': 43,
         'interviews': 43,
         'claude_api_and_console': 39,
         'general_help': 28,
         'privacy_and_legal': 20,
         'claude_code': 19,
         'claude_mobile_apps': 19,
         'skillup': 19,
         'connectors': 18,
         'pro_and_max_plans': 16,
         'safeguards': 15,
         'support': 12,
         'claude_for_government': 9,
         'engage': 9,
         'claude_desktop': 8,
         'chakra': 7,
         'amazon_bedrock': 6,
         'claude_for_nonprofits': 6,
         'claude_in_chrome': 5,
         'identity_management_sso_jit_scim': 5,
         'claude_for_education': 4,
         'uncategorized': 4,
         'index.md': 3,
         'support.md': 1})

In [32]:
#13 – Save Documents
OUTPUT_FILE = ARTIFACTS / "documents.pkl"

with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(documents, f)

print(f"Saved {len(documents)} documents to {OUTPUT_FILE}")

Saved 773 documents to C:\Users\cheru\OneDrive\Desktop\GitHub_Projects\SupportSphere-AI\artifacts\documents.pkl


In [33]:
#14 – Reload Test
with open(OUTPUT_FILE, "rb") as f:
    loaded_documents = pickle.load(f)

print(type(loaded_documents))
print(type(loaded_documents[0]))
print(len(loaded_documents))

<class 'list'>
<class 'langchain_core.documents.base.Document'>
773
